Neste notebook respondo as 10 perguntas de negócio do projeto, com Pandas.

A lógica em SQL está em [docs/insights-negocios.md](../../docs/insights-negocios.md). Cada seção traz a pergunta, o contexto de negócio e o link para a query correspondente.

In [ ]:
# A Célula de Importação
import pandas as pd
import sys

# Adiciona o diretório raiz ao caminho para o Python encontrar o conexao.py
sys.path.append('..')
from conexao import obter_dados

In [ ]:
# Carregando as tabelas principais
df_songs = obter_dados("SELECT * FROM dbo.Songs;")
df_artists = obter_dados("SELECT * FROM dbo.Artists;")
df_users = obter_dados("SELECT * FROM dbo.Users;")
df_songplays = obter_dados("SELECT * FROM dbo.SongPlays;")
df_genre = obter_dados("SELECT * FROM dbo.Genre;")
df_location = obter_dados("SELECT * FROM dbo.Location;")
df_albums = obter_dados("SELECT * FROM dbo.Albums;")
df_labels = obter_dados("SELECT * FROM dbo.Labels;")

## 1. Quais são as 10 músicas mais reproduzidas?

**Negócio:** quais conteúdos geram mais engajamento na plataforma?

**Por que importa:** base para curadoria editorial e sistemas de recomendação.

Query SQL: [insights-negocios.md - pergunta 1](../../docs/insights-negocios.md#1-quais-são-as-10-músicas-mais-reproduzidas)

In [ ]:
top_10_musicas = (
    pd.merge(df_songplays, df_songs, on='SongID', how='inner')
    .merge(df_artists, on='ArtistID', how='inner')
    .groupby(['Title', 'Name'])['SongPlayID']
    .count()
    .reset_index()
    .rename(columns={'Title': 'Musica', 'Name': 'Artista', 'SongPlayID': 'TotalReproducoes'})
    .sort_values(by='TotalReproducoes', ascending=False)
    .head(10)
)

# Exibindo o resultado final
top_10_musicas

## 2. Quais artistas acumulam mais reproduções no total?

**Negócio:** quem são os artistas âncora da plataforma?

**Por que importa:** orienta licenciamento e destaque editorial.

Query SQL: [insights-negocios.md - pergunta 2](../../docs/insights-negocios.md#2-quais-artistas-acumulam-mais-reproduções-no-total)

In [ ]:
top_10_artistas = (
    pd.merge(df_songplays, df_songs, on='SongID', how='inner')
    .merge(df_artists, on='ArtistID', how='inner')
    .groupby('Name')
    .agg(
        TotalReproducoes=('SongPlayID', 'count'),
        QtdMusicas=('SongID', 'nunique')
    )
    .reset_index()
    .rename(columns={'Name' : 'Artistas'})
    .sort_values(by='TotalReproducoes', ascending=False)
    .head(10)
)

# Exibindo o resultado no notebook
top_10_artistas

## 3. Quais gêneros musicais são mais consumidos?

**Negócio:** quais gêneros devem receber mais investimento em catálogo?

**Por que importa:** mostra onde concentrar aquisição de conteúdo.

Query SQL: [insights-negocios.md - pergunta 3](../../docs/insights-negocios.md#3-quais-gêneros-musicais-são-mais-consumidos)

In [ ]:
top_generos = (
    pd.merge(df_songplays, df_songs, on='SongID', how='inner')
    .merge(df_genre, on='GenreID', how='inner')
    .groupby('Name')
    .agg(
        TotalReproducoes=('SongPlayID', 'count'),
        QtdMusicas=('SongID', 'nunique')
    )
    .reset_index()
    .rename(columns={'Name' : 'Genero'})
    .sort_values(by='TotalReproducoes', ascending=False)
)

top_generos

## 4. Qual o volume de reproduções por mês?

**Negócio:** o engajamento está crescendo, estável ou caindo?

**Por que importa:** série temporal para tendência e sazonalidade.

Query SQL: [insights-negocios.md - pergunta 4](../../docs/insights-negocios.md#4-qual-o-volume-de-reproduções-por-mês)

In [ ]:
reproducoes_por_mes = (
    df_songplays
    .assign(
        Ano=lambda df: pd.to_datetime(df['StartTime']).dt.year,
        Mes=lambda df: pd.to_datetime(df['StartTime']).dt.month
    )
    .groupby(['Ano', 'Mes'])
    .agg(TotalReproducoes=('SongPlayID', 'count'))
    .reset_index()
    .sort_values(by=['Ano', 'Mes'])
)

reproducoes_por_mes

## 5. Quais países têm mais reproduções?

**Negócio:** onde está concentrada a audiência da plataforma?

**Por que importa:** direciona expansão e localização de conteúdo.

Query SQL: [insights-negocios.md - pergunta 5](../../docs/insights-negocios.md#5-quais-países-têm-mais-reproduções)

In [ ]:
reproducoes_por_pais=(
    pd.merge(df_songplays, df_location, on='LocationID', how='inner')
    .groupby('Country')
    .agg(
        TotalReproducoes=('SongPlayID', 'count'),
        UsuariosAtivos=('UserID', 'nunique')
    )
    .reset_index()
    .sort_values(by='TotalReproducoes', ascending=False)
)

reproducoes_por_pais

## 6. Quais álbuns têm a maior média de reproduções por faixa?

**Negócio:** quais álbuns performam de forma consistente, não só pelo hit isolado?

**Por que importa:** identifica qualidade uniforme, não só um single popular.

Query SQL: [insights-negocios.md - pergunta 6](../../docs/insights-negocios.md#6-quais-álbuns-têm-a-maior-média-de-reproduções-por-faixa)

In [ ]:
# Garantindo os nomes limpos
df_albums_limpo = df_albums.rename(columns={'Name': 'Album'})
df_artists_limpo = df_artists.rename(columns={'Name': 'Artista'})

top_10_albuns_consistentes = (
    pd.merge(df_songplays, df_songs, on='SongID', how='inner')
    .merge(df_albums_limpo, on=['AlbumID', 'ArtistID'], how='inner')
    .merge(df_artists_limpo, on='ArtistID', how='inner')
    .groupby(['Album', 'Artista'])
    .agg(
        QtdFaixas=('SongID', 'nunique'),
        TotalReproducoes=('SongPlayID', 'count')
    )
    .reset_index()
    .query('QtdFaixas >= 3')
    .assign(MediaPorFaixa=lambda df: df['TotalReproducoes'] / df['QtdFaixas'])
    .sort_values(by='MediaPorFaixa', ascending=False)
    .head(10)
)

# Exibindo o resultado no notebook
top_10_albuns_consistentes

## 7. Quais gravadoras têm mais músicas no catálogo?

**Negócio:** existe concentração de catálogo em poucas gravadoras?

**Por que importa:** concentração alta é risco contratual para a plataforma.

Query SQL: [insights-negocios.md - pergunta 7](../../docs/insights-negocios.md#7-quais-gravadoras-têm-mais-músicas-no-catálogo)

In [ ]:
# Renomeando previamente para evitar conflitos e já deixar no formato final
df_labels_limpo = df_labels.rename(columns={'Name': 'Gravadora'})

gravadoras_catalogo = (
    pd.merge(df_songs, df_albums, on='AlbumID', how='inner')
    .merge(df_labels_limpo, on='LabelID', how='inner')
    .groupby('Gravadora')
    .agg(
        QtdMusicas=('SongID', 'nunique'),
        QtdAlbuns=('AlbumID', 'nunique')
    )
    .reset_index()
    # Criando o percentual: dividindo a linha atual pela soma total da coluna
    .assign(
        PercentualCatalogo=lambda df: (df['QtdMusicas'] / df['QtdMusicas'].sum() * 100).round(2)
    )
    .sort_values(by='QtdMusicas', ascending=False)
)

# Exibindo o resultado no notebook
gravadoras_catalogo

## 8. Quantos novos usuários foram cadastrados por mês?

**Negócio:** qual é o ritmo de aquisição de usuários?

**Por que importa:** curva de crescimento da base - métrica de saúde da plataforma.

Query SQL: [insights-negocios.md - pergunta 8](../../docs/insights-negocios.md#8-quantos-novos-usuários-foram-cadastrados-por-mês)

In [ ]:
aquisicao_usuarios = (
    df_users
    .assign(
        Ano=lambda df: pd.to_datetime(df['DateCreated']).dt.year,
        Mes=lambda df: pd.to_datetime(df['DateCreated']).dt.month
    )
    .groupby(['Ano', 'Mes'])
    .agg(NovosUsuarios=('UserID', 'count'))
    .reset_index()
    .sort_values(by=['Ano', 'Mes'])
    )

aquisicao_usuarios

## 9. Qual o horário de pico de reproduções?

**Negócio:** em que horas do dia os usuários mais consomem música?

**Por que importa:** define janelas para lançamentos, notificações e campanhas.

Query SQL: [insights-negocios.md - pergunta 9](../../docs/insights-negocios.md#9-qual-o-horário-de-pico-de-reproduções)

In [ ]:
horario_pico = (
    df_songplays
    .assign(Hora=lambda df: pd.to_datetime(df['StartTime']).dt.hour)
    .groupby('Hora')
    .agg(TotalReproducoes=('SongPlayID', 'count'))
    .reset_index()
    .sort_values(by='TotalReproducoes', ascending=False)
)

horario_pico

## 10. Qual a duração média das sessões de escuta por país?

**Negócio:** usuários de quais países passam mais tempo na plataforma?

**Por que importa:** tempo de sessão mede engajamento real, não só quantidade de plays.

Query SQL: [insights-negocios.md - pergunta 10](../../docs/insights-negocios.md#10-qual-a-duração-média-das-sessões-de-escuta-por-país)

In [ ]:
duracao_sessao_pais = (
    pd.merge(df_songplays, df_location, on='LocationID', how='inner')
    .assign(
        Inicio=lambda df: pd.to_datetime(df['StartTime']),
        Fim=lambda df: pd.to_datetime(df['EndTime']),
        DuracaoMinutos=lambda df: (df['Fim'] - df['Inicio']).dt.total_seconds() / 60
    )
    .groupby('Country')
    .agg(
        TotalReproducoes=('SongPlayID', 'count'),
        DuracaoMediaMinutos=('DuracaoMinutos', 'mean')
    )
    .reset_index()
    .rename(columns= {'Country' : 'Pais'})
    .assign(DuracaoMediaMinutos=lambda df: df['DuracaoMediaMinutos'].round(2))
    .sort_values(by='DuracaoMediaMinutos', ascending=False)
)

duracao_sessao_pais